# Genomic Causal Inference — Example Analysis

This notebook walks through the pipeline interactively.

## Setup

Make sure you've installed the package:
```bash
pip install -e ".[dev]"
```

In [ ]:
import sys
sys.path.insert(0, '../src')

from genomic_causal import Config, run_pipeline
from genomic_causal.data_loader import build_dataset
from genomic_causal.feature_selection import rank_features_by_correlation
from genomic_causal.causal import run_mendelian_randomization, run_double_ml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Configure the Pipeline

Adjust paths to point to your data.

In [ ]:
cfg = Config(
    vcf_dir='../data/vcf',
    height_file='../data/GTEx-Height-Demographics.xlsx',
    output_dir='../results',
    rf_n_estimators=100,
    random_seed=2022,
)

## 2. Load Data

In [ ]:
X, Y, height_df, genotype_df, variant_ids = build_dataset(cfg)
print(f'X shape: {X.shape}')
print(f'Y shape: {Y.shape}')
print(f'Height stats: mean={Y.mean():.1f}, std={Y.std():.1f}')

## 3. Feature Ranking

In [ ]:
sorted_indices, sorted_names = rank_features_by_correlation(
    X, Y, list(genotype_df.columns)
)
print('Top 10 features:')
for i in range(10):
    print(f'  {i+1}. {sorted_names[i]}')

## 4. Run Full Pipeline

This runs all models and saves results to the output directory.

In [ ]:
run_pipeline(cfg)

## 5. Inspect Results

In [ ]:
import os

for f in sorted(os.listdir(cfg.output_dir)):
    print(f)

# Load MR results
mr_path = os.path.join(cfg.output_dir, 'MR_Results.xlsx')
if os.path.exists(mr_path):
    mr_df = pd.read_excel(mr_path)
    display(mr_df)

# Load DML results
dml_path = os.path.join(cfg.output_dir, 'DML_Results.xlsx')
if os.path.exists(dml_path):
    dml_df = pd.read_excel(dml_path)
    display(dml_df)